1. Typage des colonnes et chargement des donnees

In [1]:
import pandas as pd

dtypes = {
    "iu_ac": "string",
    "libelle": "string",
    "iu_nd_amont": "string",
    "libelle_nd_amont": "string",
    "iu_nd_aval": "string",
    "libelle_nd_aval": "string",
    "etat_trafic": "category",
}

# Chargement avec conversion des dates en UTC puis conversion en heure de Paris
df = pd.read_csv(
    "../data/raw/comptages.csv",
    sep=";",
    dtype=dtypes,
    parse_dates=["t_1h"],
)

# Gestion du fuseau horaire : on passe du fuseau UTC vers le fuseau Europe/Paris
df["t_1h"] = pd.to_datetime(df["t_1h"], utc=True).dt.tz_convert("Europe/Paris")

# Filtrage sur tes 3 axes
axes = ["Av_Foch", "Bd_Grenelle", "Bd_Barbes"]
df = df[df["libelle"].isin(axes)].copy()

print("Format des dates et types validés :")
print(df.dtypes[["t_1h", "iu_ac", "etat_trafic"]])

Format des dates et types validés :
t_1h           datetime64[us, Europe/Paris]
iu_ac                                string
etat_trafic                        category
dtype: object


2. Test et suppression des doublons

In [3]:
# 1. Nettoyage préliminaire des chaînes pour éviter les doublons cachés par espaces
df["iu_ac"] = df["iu_ac"].str.strip()

# 2. Test des doublons stricts sur la clé d'unicité naturelle
cle_unicite = ["iu_ac", "t_1h"]
nb_doublons = df.duplicated(subset=cle_unicite).sum()
print(f"Nombre de doublons stricts détectés sur (iu_ac, t_1h) : {nb_doublons}")

# 3. Test des doublons cachés (on fait un arrondi temporel à l'heure pile)
df["t_1h_ronde"] = df["t_1h"].dt.floor("h")
nb_doublons_caches = df.duplicated(subset=["iu_ac", "t_1h_ronde"]).sum()
print(f"Nombre de doublons cachés (arrondi horaire) : {nb_doublons_caches}")

# 4. Suppression si nécessaire et nettoyage de la colonne temporaire
if nb_doublons > 0 or nb_doublons_caches > 0:
    df = df.drop_duplicates(subset=["iu_ac", "t_1h_ronde"], keep="first")
    print("Doublons (stricts ou cachés) supprimés avec succès.")
else:
    print("Aucun doublon strict ni caché détecté.")

df = df.drop(columns=["t_1h_ronde"])

Nombre de doublons stricts détectés sur (iu_ac, t_1h) : 0
Nombre de doublons cachés (arrondi horaire) : 0
Aucun doublon strict ni caché détecté.


3. Suppression des colonnes inutiles avec justification par test

In [5]:
cols_a_supprimer = []

# TEST 1 : Détection des colonnes à valeur unique ou 100% vides
print("TEST 1 : Colonnes constantes ou vides")
for col in df.columns:
    nb_valeurs = df[col].nunique(dropna=False)
    if nb_valeurs <= 1:
        cols_a_supprimer.append(col)
        val_unique = df[col].unique()
        print(f"-> Suppression de '{col}' : test val_unique <= 1 validé (Valeur : {val_unique})")

#TEST 2 : Preuve de la redondance des nœuds amont/aval
print("\nTEST 2 : Justification des nœuds amont/aval")

redundant_cols = ["iu_nd_amont", "libelle_nd_amont", "iu_nd_aval", "libelle_nd_aval"]

for col in redundant_cols:
    if col in df.columns and col not in cols_a_supprimer:
        # Test de correspondance : on vérifie que chaque iu_ac a une relation fixe avec ces nœuds
        # c'est la preuve que ce sont des sous-attributs du capteur iu_ac et non d'autres variables de trafic)
        nb_relations = df.groupby("iu_ac")[col].nunique().max()

        #On retire les noeuds amont/aval (libelle_nd_amont / libelle_nd_aval) parce qu'ils n'ont aucune utilité pour l'analyse ici
        cols_a_supprimer.append(col)
        print(f"-> Suppression de '{col}' : test de redondance validé ({nb_relations} valeur(s) par iu_ac). "
              f"Il s'agit d'une information secondaire non nécessaire pour l'analyse globale.")

#APPLICATION DE LA SUPPRESSION
df = df.drop(columns=cols_a_supprimer)

print(f"\nTotal : {len(cols_a_supprimer)} colonne(s) supprimée(s).")
print(f"Colonnes restantes dans le DataFrame : {list(df.columns)}")

TEST 1 : Colonnes constantes ou vides

TEST 2 : Justification des nœuds amont/aval

Total : 0 colonne(s) supprimée(s).
Colonnes restantes dans le DataFrame : ['iu_ac', 'libelle', 't_1h', 'q', 'k', 'etat_trafic', 'etat_barre']


4. Analyse des valeurs manquantes et traitement des trous

In [ ]:
import numpy as np
import pandas as pd

#pour empecher pandas de tronquer les lignes
pd.set_option('display.max_rows', 20)

#EXPLORATION : Analyse de la réalité physique des états d'anomalie

# 1. Détection automatique de la colonne contenant l'info ("etat_barre" ou "etat_trafic")
col_etat = "etat_barre" if "etat_barre" in df.columns else "etat_trafic"

# 2. Isolation des anomalies (nous gerons les codes 2/3 de etat_barre ou les textes Barré/Invalide/Inconnu)
masque_anomalies = df[col_etat].isin([2, 3, "Barré", "Invalide", "2", "3"])
anomalies = df[masque_anomalies]

# 3. Calcul du volume et du débit moyen observé malgré le statut critique
analyse_terrain = anomalies.groupby(["libelle", col_etat]).agg(
    nb_heures=("q", "count"),
    heures_avec_trafic=("q", lambda x: (x > 0).sum()),
    debit_moyen=("q", "mean"),
    debit_max=("q", "max")
).reset_index()

print("Réalité du trafic mesuré lors des états d'anomalie ")
print(analyse_terrain)


# Signification et passage des données non fiables en NaN
print("\n--- 1. Répartition des états particuliers par axe")
print(pd.crosstab(df["libelle"], df[col_etat], dropna=False))

# Passage explicite des valeurs non fiables en NaN
df.loc[masque_anomalies, ["q", "k"]] = np.nan


# 2. Vraie analyse de la longueur des trous de données
def calculer_longueur_trous(serie_valeurs):
    is_null = serie_valeurs.isna()
    groupes = (~is_null).cumsum()[is_null]
    if groupes.empty:
        return pd.Series(dtype=int)
    return groupes.value_counts()

longueurs_trous = df.groupby("iu_ac")["q"].apply(calculer_longueur_trous)

print("\n2. Analyse de la longueur des trous de données (en heures)")
if not longueurs_trous.empty:
    print(longueurs_trous.describe())
else:
    print("Aucun trou de données détecté.")


# 3. Methode choisie: Imputation ciblée : pour les trous courts (<= 2h) 
# si par exemple un capteur est indisponible sur un temps < 2h, au dela ce serait trop incertain 

df["q"] = df.groupby("iu_ac")["q"].transform(lambda x: x.interpolate(method="linear", limit=2))
df["k"] = df.groupby("iu_ac")["k"].transform(lambda x: x.interpolate(method="linear", limit=2))

print("\nImputation effectuée avec succès (interpolation linéaire, limit=2).")

Réalité du trafic mesuré lors des états d'anomalie 
       libelle etat_barre  nb_heures  heures_avec_trafic  debit_moyen  \
0      Av_Foch      Barré        171                 103     53.74269   
1      Av_Foch   Invalide          0                   0          NaN   
2    Bd_Barbes   Invalide          0                   0          NaN   
3  Bd_Grenelle   Invalide          0                   0          NaN   

   debit_max  
0      792.0  
1        NaN  
2        NaN  
3        NaN  

--- 1. Répartition des états particuliers par axe
etat_barre   Barré  Invalide  Ouvert
libelle                             
Av_Foch        498     12731   39139
Bd_Barbes        0      1933   50435
Bd_Grenelle      0      1080   42560

2. Analyse de la longueur des trous de données (en heures)
count    1829.000000
mean       16.180973
std       217.740756
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max      4364.000000
Name: q, dtype: float64

Imputation effectu

Invalide : Correspond à une panne matérielle ou un décalibrage de la boucle électromagnétique. Le capteur est hors-service ou transmet des données incohérentes.   
Barré : Correspond à une fermeture administrative de la voie en question

interpretation de la Ligne 0: Elle prouve qu'il y a un décalage entre la fermeture administrative de la voie (chantier ou autre événement) et la réalité physique où des usagers  continuent de circuler, ou que les capteurs continuent d'enregistrer des flux de facon résiduelle malgré les barrières.

5. Export

In [8]:
import os

os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/comptages_prepares.csv", index=False, sep=";")

print("Données nettoyées exportées dans data/processed/comptages_prepares.csv")

Données nettoyées exportées dans data/processed/comptages_prepares.csv
